In [3]:
import pandas as pd

df = pd.read_csv("./synthetic_logs.csv")
df.head()

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert


In [4]:
df.source.unique()
df.target_label.unique()

array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'System Notification', 'Resource Usage', 'User Action',
       'Workflow Error', 'Deprecation Warning'], dtype=object)

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

embeddings = model.encode(df['log_message'].tolist())

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7685.57it/s]


In [6]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.25, min_samples=2, metric="cosine")
clusters = dbscan.fit_predict(embeddings)

df['cluster'] = clusters

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [7]:
import re

def classify_regex(log_message):
  PATTERNS = {
    "HTTP Status":          r'nova\.\w+\.wsgi\.server\s+(?:[-\]]|\[req-[a-f0-9-]+[^\]]*\])\s+[\d.,]+\s+"(?:GET|POST|PUT|DELETE|PATCH)\s+\S+\s+HTTP/\d\.\d"',
    "Resource Usage":       r'nova\.compute\.(?:claims|resource_tracker)\s+\[req-[a-f0-9-]+',
    "Security Alert":       r'(?:Unauthorized access|bad login|brute force|login attempt|Suspicious login|Denied access|security breach|bypass.*?security|login failures|Privilege elevation|not authorized|intrusion detection|privilege misuse|security threat|admin privileges detected|API security.*?(?:suspicious|detected)|admin access escalation|failed to provide valid.*?credentials|Security alert:.*?suspicious|unusual system calls|escalated admin privileges)',
    "System Notification":  r'(?:uploaded successfully|Backup (?:completed|started|ended)|maintenance|System update|patch applied|reboot initiated|Disk cleanup|scheduled)',
    "User Action":          r'(?:Account with ID \d+ (?:created|deleted|updated) by|User \S+ logged (?:in|out)|Password (?:for user|changed|reset))',
    "Error":                r'(?:replication task|server encountered|sending fault|did not complete|restarted without warning|crashed unexpectedly|Unexpected.*?stoppage|delivery glitch|problem sending|unplanned restart|abrupt restart|health check.*?not successful|SSL certificate|Replication of data.*?failed|data copy failed|Abnormal shutdown|replication.*?encountered|synchronization task failed|invalid data format|replication.*?unsuccessful|replication failure)',
    "Critical Error":       r'(?:Email service experiencing issues|Critical system unit error|multiple disk faults|RAID|database.*?failure|service.*?down|outage|component malfunction|Boot process terminated|configuration.*?(?:invalid|compromised)|system element crashed|Unrecoverable|Non-recoverable|Essential system part|Service disruption.*?email|critical system crash|Critical system element is down|Failure.*?critical system component|system component.*?stopped working|Vital system component|Critical failure.*?application)',
    "Workflow Error":       r'(?:Lead conversion failed|follow-up process.*?failed|Escalation rule.*?failed|workflow.*?failed|conversion failed for prospect)',
    "Deprecation Warning":  r'(?:deprecated and will be removed|outdated.*?migrate|will be discontinued)',
}

  for label, pattern in PATTERNS.items():
    if re.search(pattern, log_message):
      return label
  return None

In [8]:
df['regex_label'] = df['log_message'].apply(classify_regex)

In [9]:
df_unknown = df[df['regex_label'].isna()].copy()

In [10]:
X = model.encode(
    df_unknown['log_message'].tolist(),
    show_progress_bar=True
)

y = df_unknown['target_label']
y.value_counts()

Batches: 100%|██████████| 17/17 [00:00<00:00, 33.55it/s]


target_label
Security Alert    235
Critical Error    104
Error             100
HTTP Status        89
Workflow Error      1
Name: count, dtype: int64

In [11]:
class_counts = y.value_counts()
rare_classes = class_counts[class_counts < 2].index

mask = ~y.isin(rare_classes)

X = X[mask]
y = y[mask]

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [13]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

clf.fit(X_train, y_train)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/L

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [14]:
from sklearn.metrics import classification_report

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

                precision    recall  f1-score   support

Critical Error       1.00      1.00      1.00        21
         Error       1.00      0.95      0.97        20
   HTTP Status       1.00      1.00      1.00        18
Security Alert       0.98      1.00      0.99        47

      accuracy                           0.99       106
     macro avg       0.99      0.99      0.99       106
  weighted avg       0.99      0.99      0.99       106



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [15]:
CONFIDENCE_THRESHOLD = 0.70

def classify_with_threshold(log_message):
    # Step 1: Try regex first
    regex_label = classify_regex(log_message)
    if regex_label:
        return regex_label, 'regex', 1.0

    # Step 2: Embed the log and get BERT probabilities
    embedding  = model.encode([log_message])
    probs      = clf.predict_proba(embedding)[0]
    confidence = probs.max()
    bert_label = clf.classes_[probs.argmax()]

    # Step 3: Confidence gate
    if confidence >= CONFIDENCE_THRESHOLD:
        return bert_label, 'bert', confidence
    else:
        return 'NEEDS_LLM', 'llm', confidence


In [16]:
import joblib

joblib.dump(clf, "model/classifier.joblib")

['model/classifier.joblib']